In [0]:
spark.sql("""
CREATE OR REPLACE TABLE 03_gold_catalog.gold_schema.curated_kpi_metrics AS
WITH sales_base AS (
    SELECT 
        transaction_key,
        customer_key,
        product_key,
        store_key,
        transaction_date,
        quantity_sold,
        unit_price,
        total_sales_amount,
        discount_amount,
        profit_amount,
        product_name,
        product_category
    FROM 03_gold_catalog.gold_schema.fact_sales
    WHERE transaction_date IS NOT NULL
),
returns_agg AS (
    SELECT 
        transaction_key,
        COUNT(*) as return_count,
        SUM(returned_amount) as total_returned_amount
    FROM 03_gold_catalog.gold_schema.fact_returns
    GROUP BY transaction_key
),
inventory_current AS (
    SELECT 
        product_key,
        store_key,
        stock_quantity,
        is_out_of_stock,
        inventory_value_cost,
        inventory_value_retail
    FROM 03_gold_catalog.gold_schema.fact_inventory
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_key, store_key ORDER BY snapshot_date DESC) = 1
)
SELECT 
    -- Transaction identifiers
    s.transaction_key,
    s.customer_key,
    s.product_key,
    s.store_key,
    s.transaction_date,
    
    -- Date components for time-based analysis
    YEAR(s.transaction_date) as transaction_year,
    MONTH(s.transaction_date) as transaction_month,
    DAY(s.transaction_date) as transaction_day,
    DATE_TRUNC('day', s.transaction_date) as transaction_date_only,
    
    -- Product attributes
    s.product_name,
    s.product_category,
    
    -- Sales metrics
    s.quantity_sold,
    s.unit_price,
    s.total_sales_amount,
    s.discount_amount,
    s.profit_amount,
    
    -- Return metrics
    COALESCE(r.return_count, 0) as return_count,
    COALESCE(r.total_returned_amount, 0) as returned_amount,
    CASE WHEN r.transaction_key IS NOT NULL THEN 1 ELSE 0 END as is_returned,
    
    -- Inventory metrics (current snapshot)
    COALESCE(i.stock_quantity, 0) as current_stock_quantity,
    COALESCE(i.is_out_of_stock, 0) as is_out_of_stock,
    COALESCE(i.inventory_value_cost, 0) as inventory_value_cost,
    COALESCE(i.inventory_value_retail, 0) as inventory_value_retail,
    
    -- Customer metrics flags
    1 as transaction_count  -- for counting transactions per customer
    
FROM sales_base s
LEFT JOIN returns_agg r ON s.transaction_key = r.transaction_key
LEFT JOIN inventory_current i ON s.product_key = i.product_key AND s.store_key = i.store_key
""")


In [0]:
result = spark.sql("""
SELECT 
    CONCAT('$', FORMAT_NUMBER(SUM(total_sales_amount), 2)) as total_sales,
    FORMAT_NUMBER(COUNT(DISTINCT transaction_key), 0) as total_transactions,
    FORMAT_NUMBER(COUNT(DISTINCT customer_key), 0) as unique_customers
FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
""")

display(result)

In [0]:
result = spark.sql("""
SELECT 
    product_name,
    product_category,
    FORMAT_NUMBER(SUM(quantity_sold), 0) as units_sold,
    CONCAT('$', FORMAT_NUMBER(SUM(total_sales_amount), 2)) as total_revenue,
    CONCAT('$', FORMAT_NUMBER(SUM(profit_amount), 2)) as total_profit,
    CONCAT('$', FORMAT_NUMBER(AVG(profit_amount / quantity_sold), 2)) as avg_profit_per_unit
FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
GROUP BY product_name, product_category
ORDER BY SUM(profit_amount) DESC
LIMIT 10
""")

display(result)

In [0]:
result = spark.sql("""
WITH category_sales AS (
    SELECT 
        product_category,
        SUM(total_sales_amount) as total_revenue,
        SUM(quantity_sold) as units_sold,
        COUNT(DISTINCT transaction_key) as transactions,
        SUM(profit_amount) as total_profit
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    GROUP BY product_category
)
SELECT 
    product_category,
    CONCAT('$', FORMAT_NUMBER(total_revenue, 2)) as total_revenue,
    FORMAT_NUMBER(units_sold, 0) as units_sold,
    FORMAT_NUMBER(transactions, 0) as transactions,
    CONCAT('$', FORMAT_NUMBER(total_profit, 2)) as total_profit,
    CONCAT(ROUND(total_revenue * 100.0 / SUM(total_revenue) OVER(), 2), '%') as revenue_share
FROM category_sales
ORDER BY total_revenue DESC
""")

display(result)

In [0]:
result = spark.sql("""
WITH transaction_items AS (
    SELECT 
        transaction_key,
        SUM(quantity_sold) as total_items_in_basket
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    GROUP BY transaction_key
)
SELECT 
    ROUND(AVG(total_items_in_basket), 2) as avg_basket_size,
    MIN(total_items_in_basket) as min_basket_size,
    MAX(total_items_in_basket) as max_basket_size,
    FORMAT_NUMBER(COUNT(transaction_key), 0) as total_transactions
FROM transaction_items
""")

display(result)

In [0]:
result = spark.sql("""
SELECT 
    FORMAT_NUMBER(COUNT(DISTINCT CASE WHEN is_returned = 1 THEN transaction_key END), 0) as returned_transactions,
    FORMAT_NUMBER(COUNT(DISTINCT transaction_key), 0) as total_transactions,
    CONCAT(ROUND(COUNT(DISTINCT CASE WHEN is_returned = 1 THEN transaction_key END) * 100.0 / COUNT(DISTINCT transaction_key), 2), '%') as return_rate_by_transactions,
    CONCAT('$', FORMAT_NUMBER(SUM(returned_amount), 2)) as total_returned_amount,
    CONCAT('$', FORMAT_NUMBER(SUM(total_sales_amount), 2)) as total_sales_amount,
    CONCAT(ROUND(SUM(returned_amount) * 100.0 / NULLIF(SUM(total_sales_amount), 0), 2), '%') as return_rate_by_amount
FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
""")

display(result)

In [0]:
result = spark.sql("""
SELECT 
    FORMAT_NUMBER(COUNT(DISTINCT CASE WHEN is_out_of_stock = 1 THEN product_key END), 0) as out_of_stock_products,
    FORMAT_NUMBER(COUNT(DISTINCT product_key), 0) as total_products,
    CONCAT(ROUND(COUNT(DISTINCT CASE WHEN is_out_of_stock = 1 THEN product_key END) * 100.0 / COUNT(DISTINCT product_key), 2), '%') as out_of_stock_rate
FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
""")

display(result)

print("\n\u26a0️ Products currently out of stock:")
out_of_stock_details = spark.sql("""
SELECT DISTINCT
    product_name,
    product_category,
    current_stock_quantity
FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
WHERE is_out_of_stock = 1
ORDER BY product_category, product_name
""")
display(out_of_stock_details)

In [0]:
result = spark.sql("""
WITH daily_store_sales AS (
    SELECT 
        m.store_key,
        d.store_name,
        d.store_region,
        m.transaction_date_only,
        SUM(m.total_sales_amount) as daily_sales
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics m
    JOIN 03_gold_catalog.gold_schema.dim_store d ON m.store_key = d.store_key
    GROUP BY m.store_key, d.store_name, d.store_region, m.transaction_date_only
),
store_aggregates AS (
    SELECT 
        store_key,
        store_name,
        store_region,
        AVG(daily_sales) as avg_daily_sales,
        SUM(daily_sales) as total_sales,
        COUNT(DISTINCT transaction_date_only) as days_active
    FROM daily_store_sales
    GROUP BY store_key, store_name, store_region
)
SELECT 
    store_name,
    store_region,
    CONCAT('$', FORMAT_NUMBER(avg_daily_sales, 2)) as avg_daily_sales,
    CONCAT('$', FORMAT_NUMBER(total_sales, 2)) as total_sales,
    FORMAT_NUMBER(days_active, 0) as days_active,
    DENSE_RANK() OVER (ORDER BY avg_daily_sales DESC) as performance_rank
FROM store_aggregates
ORDER BY avg_daily_sales DESC
LIMIT 5
""")

display(result)

In [0]:

result = spark.sql("""
SELECT 
    CONCAT('$', FORMAT_NUMBER(SUM(discount_amount), 2)) as total_discount_given,
    CONCAT('$', FORMAT_NUMBER(SUM(total_sales_amount), 2)) as total_sales_amount,
    CONCAT('$', FORMAT_NUMBER(SUM(total_sales_amount + discount_amount), 2)) as potential_revenue_without_discount,
    CONCAT(ROUND(SUM(discount_amount) * 100.0 / NULLIF(SUM(total_sales_amount + discount_amount), 0), 2), '%') as discount_rate,
    FORMAT_NUMBER(COUNT(DISTINCT transaction_key), 0) as transactions_with_discount
FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
WHERE discount_amount > 0
""")

display(result)

print("\nDiscount impact by product category:")
category_discount = spark.sql("""
SELECT 
    product_category,
    CONCAT('$', FORMAT_NUMBER(SUM(discount_amount), 2)) as total_discount,
    CONCAT('$', FORMAT_NUMBER(SUM(total_sales_amount), 2)) as total_sales,
    CONCAT(ROUND(SUM(discount_amount) * 100.0 / NULLIF(SUM(total_sales_amount + discount_amount), 0), 2), '%') as discount_rate
FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
WHERE discount_amount > 0
GROUP BY product_category
ORDER BY SUM(discount_amount) DESC
""")
display(category_discount)

In [0]:

result = spark.sql("""
WITH customer_purchase_count AS (
    SELECT 
        customer_key,
        COUNT(DISTINCT transaction_key) as purchase_count
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    GROUP BY customer_key
)
SELECT 
    FORMAT_NUMBER(COUNT(CASE WHEN purchase_count > 1 THEN customer_key END), 0) as repeat_customers,
    FORMAT_NUMBER(COUNT(customer_key), 0) as total_customers,
    CONCAT(ROUND(COUNT(CASE WHEN purchase_count > 1 THEN customer_key END) * 100.0 / COUNT(customer_key), 2), '%') as repeat_customer_rate,
    ROUND(AVG(CASE WHEN purchase_count > 1 THEN purchase_count END), 2) as avg_purchases_per_repeat_customer
FROM customer_purchase_count
""")

display(result)

print("\nCustomer segmentation by purchase frequency:")
segmentation = spark.sql("""
WITH customer_purchase_count AS (
    SELECT 
        customer_key,
        COUNT(DISTINCT transaction_key) as purchase_count
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    GROUP BY customer_key
)
SELECT 
    CASE 
        WHEN purchase_count = 1 THEN '1 purchase (One-time)'
        WHEN purchase_count BETWEEN 2 AND 5 THEN '2-5 purchases (Regular)'
        WHEN purchase_count BETWEEN 6 AND 10 THEN '6-10 purchases (Loyal)'
        ELSE '11+ purchases (VIP)'
    END as customer_segment,
    FORMAT_NUMBER(COUNT(customer_key), 0) as customer_count,
    CONCAT(ROUND(COUNT(customer_key) * 100.0 / SUM(COUNT(customer_key)) OVER(), 2), '%') as percentage
FROM customer_purchase_count
GROUP BY 
    CASE 
        WHEN purchase_count = 1 THEN '1 purchase (One-time)'
        WHEN purchase_count BETWEEN 2 AND 5 THEN '2-5 purchases (Regular)'
        WHEN purchase_count BETWEEN 6 AND 10 THEN '6-10 purchases (Loyal)'
        ELSE '11+ purchases (VIP)'
    END
ORDER BY 
    CASE 
        WHEN customer_segment = '1 purchase (One-time)' THEN 1
        WHEN customer_segment = '2-5 purchases (Regular)' THEN 2
        WHEN customer_segment = '6-10 purchases (Loyal)' THEN 3
        ELSE 4
    END
""")
display(segmentation)

In [0]:
result = spark.sql("""
WITH recent_sales AS (
    SELECT DISTINCT product_key
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    WHERE transaction_date >= CURRENT_DATE() - INTERVAL 30 DAYS
),
all_products_with_inventory AS (
    SELECT DISTINCT 
        product_key,
        product_name,
        product_category,
        current_stock_quantity
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    WHERE current_stock_quantity > 0
)
SELECT 
    a.product_name,
    a.product_category,
    a.current_stock_quantity,
    'No sales in last 30 days' as status
FROM all_products_with_inventory a
LEFT JOIN recent_sales r ON a.product_key = r.product_key
WHERE r.product_key IS NULL
ORDER BY a.current_stock_quantity DESC, a.product_category, a.product_name
""")

display(result)

print("\nSummary:")
summary = spark.sql("""
WITH recent_sales AS (
    SELECT DISTINCT product_key
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    WHERE transaction_date >= CURRENT_DATE() - INTERVAL 30 DAYS
),
all_products_with_inventory AS (
    SELECT DISTINCT product_key
    FROM 03_gold_catalog.gold_schema.curated_kpi_metrics
    WHERE current_stock_quantity > 0
)
SELECT 
    FORMAT_NUMBER(COUNT(a.product_key), 0) as slow_moving_products,
    FORMAT_NUMBER((SELECT COUNT(DISTINCT product_key) FROM all_products_with_inventory), 0) as total_products_in_stock,
    CONCAT(ROUND(COUNT(a.product_key) * 100.0 / (SELECT COUNT(DISTINCT product_key) FROM all_products_with_inventory), 2), '%') as slow_moving_rate
FROM all_products_with_inventory a
LEFT JOIN recent_sales r ON a.product_key = r.product_key
WHERE r.product_key IS NULL
""")
display(summary)